# Seeded `PPM.assign_rnd()` demonstration

This notebook demonstrates the Python counterpart of C++ `PPM::AssignRnd(int num_digs)`. The method generates a decimal digit string and assigns it through the ordinary `PPM.assign()` path. A fixed seed makes this notebook repeatable.

In [1]:
from pathlib import Path
import sys


def find_repo_root(start=None):
    path = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (path, *path.parents):
        if (candidate / "rns_pypal").exists() and (candidate / "tests").exists():
            return candidate
    raise RuntimeError("Could not find the RNS-PyPAL repository root")


repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

repo_root

WindowsPath('d:/Projects/RNS-APAL-REPO/RNS-APAL')

## Specify a larger power-based RNS system

The six pairwise-coprime base moduli are the primes from 2 through 13. Each RNS digit uses `base ** power` as its full modulus. The selected powers keep the six full digit moduli at roughly similar sizes and provide a 60-decimal-digit unsigned dynamic range.

In [2]:
from random import Random

from rns_pypal import PPM, RNSNumberSystem


BASE_MODULI = [2, 3, 5, 7, 11, 13]
POWERS = [32, 20, 14, 12, 10, 9]

system = RNSNumberSystem(
    moduli=BASE_MODULI,
    powers=POWERS,
    fractional_digits=0,
    name="power-based-through-13",
)

print(f"System: {system.name}")
print("index  base  power  full modulus")
for index, (base, power, full_modulus) in enumerate(
    zip(system.moduli, system.powers, system.full_moduli)
):
    print(f"{index:>5}  {base:>4}  {power:>5}  {full_modulus:>12}")
print(f"\nUnsigned dynamic range: 0 through {system.dynamic_range - 1}")
print(f"Dynamic-range decimal digits: {len(str(system.dynamic_range - 1))}")

System: power-based-through-13
index  base  power  full modulus
    0     2     32    4294967296
    1     3     20    3486784401
    2     5     14    6103515625
    3     7     12   13841287201
    4    11     10   25937424601
    5    13      9   10604499373

Unsigned dynamic range: 0 through 347983393392222505593118123392881663724041011199999999999999
Dynamic-range decimal digits: 60


## Assign a reproducible 50-digit random value

`assign_rnd()` mutates an existing `PPM`, matching the C++ method style. The seed is optional in ordinary use, but supplying it makes examples and tests reproducible.

In [3]:
NUM_DECIMAL_DIGITS = 50
SEED = 20_260_726

sample = PPM(0, system=system)
sample.assign_rnd(NUM_DECIMAL_DIGITS, seed=SEED)

print("Converted decimal value:")
print(sample.format_value())
print("\nNative residue digits:")
print(sample.format_native())
print("\nResidues with full-modulus header:")
print(sample.format_whdr())

Converted decimal value:
16532417518269081220223169124236009847500445006010

Native residue digits:
1688418490 3318740618 2886412260 7799438191 16644543683 8742395726

Residues with full-modulus header:
4294967296 3486784401 6103515625 13841287201 25937424601 10604499373
---------- ---------- ---------- ----------- ----------- -----------
1688418490 3318740618 2886412260 7799438191  16644543683 8742395726 


## Verify the result independently at the conversion boundary

The expected decimal text is regenerated from the same explicit seed. Expected residues are then calculated one decimal digit at a time for each full modulus. This verification is a notebook/test oracle; `PPM.assign_rnd()` itself delegates to the library's residue-domain string assignment.

In [4]:
oracle_rng = Random(SEED)
generated_text = "".join(
    str(oracle_rng.randrange(10)) for _ in range(NUM_DECIMAL_DIGITS)
)
expected_value_text = generated_text.lstrip("0") or "0"

expected_residues = []
for modulus in system.full_moduli:
    residue = 0
    for character in generated_text:
        residue = (residue * 10 + int(character)) % modulus
    expected_residues.append(residue)

assert len(generated_text) == NUM_DECIMAL_DIGITS
assert sample.format_value() == expected_value_text
assert sample.to_residues() == tuple(expected_residues)
assert not sample.any_part_skips()

print(f"Generated decimal text: {generated_text}")
print(f"Verified residues: {sample.to_residues()}")
print("All assignment and normalized-format checks passed.")

Generated decimal text: 16532417518269081220223169124236009847500445006010
Verified residues: (1688418490, 3318740618, 2886412260, 7799438191, 16644543683, 8742395726)
All assignment and normalized-format checks passed.


## Confirm seed repeatability

In [5]:
repeated = PPM(0, system=system)
repeated.assign_rnd(NUM_DECIMAL_DIGITS, seed=SEED)

different = PPM(0, system=system)
different.assign_rnd(NUM_DECIMAL_DIGITS, seed=SEED + 1)

assert repeated == sample
assert different != sample

print("Same seed reproduces the same RNS value:", repeated == sample)
print("Different seed produced a different RNS value:", different != sample)

Same seed reproduces the same RNS value: True
Different seed produced a different RNS value: True


## Usage notes

- `num_digits` must be a positive integer.
- Omit `seed` when repeatability is not required: `value.assign_rnd(50)`.
- The generator is pseudorandom and is not intended for cryptographic use.
- Generated leading zeroes are accepted, matching the C++ routine.
- Assignment restores the destination to its normalized system.
- Like ordinary unsigned `PPM.assign()`, values larger than the declared dynamic range wrap in the RNS system.